<a href="https://colab.research.google.com/github/Nawaf-Alorabi/Tw_ML_Predicting_Employee_Burnout_with_Regression/blob/main/Project_Final_Burnout.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Phase One: Data Loading & Core Integration
## Overview:
The primary goal of this phase is to establish a robust and clean foundation for our predictive study. Given the large scale of the HuggingFace synthetic dataset (~850k records), we performed a Strategic Feature Selection to focus only on variables that have a theoretical and empirical impact on employee burnout. This stage ensures our data is structurally sound, normalized in its naming conventions, and prepared for deep exploratory analysis.

## Key Objectives:
- **Selective Data Ingestion:** Streamlining the dataset by importing only **11** critical features (out of **31**) to optimize memory usage and eliminate noise during the training phase.

- **Target Standardization:** Renaming the primary metric from burnout_risk to burnout_score to align with the project's goal of predicting a continuous burnout magnitude.

- **Data Integrity Verification:** Executing a multi step audit including shape, info, and describe to validate data types and understand the statistical range of behavioral metrics.

- **Cleaning & Quality Assurance:** Conducting a rigorous check for missing values (isnull) and duplicated entries to ensure the model isn't trained on biased or redundant information.

In [ ]:
import pandas as pd
import numpy as np


In [ ]:
# URL Link for dataset
url = "https://huggingface.co/datasets/BrotherTony/employee-burnout-turnover-prediction-800k/resolve/main/synthetic-employee-dataset.json"

# Identifying the features selected + the Target (burnout_risk)
selected_columns = [
    'workload_score', 'overtime_hours', 'role_complexity_score',
    'stress_level', 'satisfaction_score', 'team_sentiment',
    'career_progression_score', 'goal_achievement_rate',
    'meeting_participation', 'role', 'burnout_risk'
]


In [ ]:
# Loading the data
BO_data = pd.read_json(url)

In [ ]:
#Filtering the data to only the selected columns
BO_data = BO_data[selected_columns]

In [ ]:
#Renaming burnout_risk to burnout_score
BO_data.rename(columns={'burnout_risk': 'burnout_score'}, inplace=True)

In [ ]:
BO_data.head()

,workload_score,overtime_hours,role_complexity_score,stress_level,satisfaction_score,team_sentiment,career_progression_score,goal_achievement_rate,meeting_participation,role,burnout_score
0,0.758117,0.00000,0.2,0.908992,0.623746,0.662335,1.000000,0.632482,0.492131,,0.866643
1,0.788416,0.00000,0.2,0.363321,0.982556,0.934661,1.000000,0.538587,0.981394,Customer Success Manager,0.218996
2,0.697617,0.00000,0.2,0.664378,0.767200,0.888559,0.836495,0.624656,0.701138,Administrative Assistant,0.541531
3,0.493143,9.59168,0.2,1.000000,0.185888,0.732189,1.000000,0.959320,0.339626,Senior Manager,1.000000
4,0.567230,0.00000,0.2,0.723049,0.566706,0.817545,1.000000,0.677305,0.565582,Anonymous Employee,0.614825


In [ ]:
BO_data.shape

(849999, 11)

In [ ]:
BO_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 849999 entries, 0 to 849998
Data columns (total 11 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   workload_score            849999 non-null  float64
 1   overtime_hours            849999 non-null  float64
 2   role_complexity_score     849999 non-null  float64
 3   stress_level              849999 non-null  float64
 4   satisfaction_score        849999 non-null  float64
 5   team_sentiment            849999 non-null  float64
 6   career_progression_score  849999 non-null  float64
 7   goal_achievement_rate     849999 non-null  float64
 8   meeting_participation     849999 non-null  float64
 9   role                      849999 non-null  object 
 10  burnout_score             849999 non-null  float64
dtypes: float64(10), object(1)
memory usage: 71.3+ MB


In [ ]:
# chaeck for missing value
BO_data.isnull().sum()

,0
workload_score,0
overtime_hours,0
role_complexity_score,0
stress_level,0
satisfaction_score,0
team_sentiment,0
career_progression_score,0
goal_achievement_rate,0
meeting_participation,0
role,0


In [ ]:
# Checking for missing duplicates
BO_data.duplicated().sum()

np.int64(0)

In [ ]:

BO_data.describe()

,workload_score,overtime_hours,role_complexity_score,stress_level,satisfaction_score,team_sentiment,career_progression_score,goal_achievement_rate,meeting_participation,burnout_score
count,849999.000000,849999.000000,849999.000000,849999.000000,849999.000000,849999.000000,849999.000000,849999.000000,849999.000000,849999.000000
mean,0.599490,3.132903,0.210859,0.791048,0.581941,0.666848,0.841147,0.692027,0.473380,0.735941
std,0.200038,6.094207,0.085018,0.264536,0.276268,0.178232,0.182216,0.183911,0.184961,0.315756
min,0.006741,0.000000,0.200000,0.000000,0.050000,0.019774,0.182198,0.074113,0.008674,0.000000
25%,0.455657,0.000000,0.200000,0.613005,0.367016,0.545729,0.713997,0.571340,0.332366,0.479950
50%,0.613744,0.000000,0.200000,0.942151,0.583188,0.686320,0.911793,0.699235,0.457102,0.914494
75%,0.756693,4.161240,0.200000,1.000000,0.813912,0.806540,1.000000,0.825585,0.600885,1.000000
max,0.999615,73.953574,1.000000,1.000000,1.000000,0.999956,1.000000,1.000000,1.000000,1.000000


In [ ]:
# Summary of Categorical (Non-numeric) Columns
BO_data.describe(include='object')

,role
count,849999
unique,300
top,Anonymous Employee
freq,224117


# Phase Two: Data Refinement & Feature Engineering

## Overview:
In this phase, we transition from data discovery to data preparation. Our goal is to refine the dataset by addressing multi-collinearity, encoding categorical variables into a machine-readable format, and scaling numerical features. This ensures that our predictive models can process the information efficiently without bias or data leakage.

## Key Objectives:
- **Feature Pruning (Leakage Prevention):** Eliminating highly correlated features (like **stress_level**) that could artificially inflate model performance.

- **Categorical Encoding:** Converting the role column into numerical values using Target Encoding to manage **300**+ unique job titles without memory overhead.

- **Feature Scaling:** Normalizing numerical ranges using StandardScaler to bring features with different units (**hours** vs. **scores**) into a comparable scale.

In [ ]:
# 1. Dropping High-Leakage Features
BO_data.drop(columns=['stress_level'], inplace=True)

In [ ]:
# to see Correlation matrix bettween the features
BO_data.corr(numeric_only=True)['burnout_score'].sort_values(ascending=False)

,burnout_score
burnout_score,1.000000
overtime_hours,0.240957
role_complexity_score,0.004662
team_sentiment,0.002773
career_progression_score,-0.011056
goal_achievement_rate,-0.106267
workload_score,-0.403526
meeting_participation,-0.471630
satisfaction_score,-0.675306


In [ ]:
# 2. Handling Categorical Variables (Target Encoding for 'role')
# Since 'role' has 300+ unique values, One-Hot Encoding is too heavy.
# We use the mean of the target for each role.
role_target_mean = BO_data.groupby('role')['burnout_score'].mean()
BO_data['role_encoded'] = BO_data['role'].map(role_target_mean)

In [ ]:
BO_data.head()

,workload_score,overtime_hours,role_complexity_score,satisfaction_score,team_sentiment,career_progression_score,goal_achievement_rate,meeting_participation,role,burnout_score,role_encoded
0,0.758117,0.00000,0.2,0.623746,0.662335,1.000000,0.632482,0.492131,,0.866643,0.728387
1,0.788416,0.00000,0.2,0.982556,0.934661,1.000000,0.538587,0.981394,Customer Success Manager,0.218996,0.756589
2,0.697617,0.00000,0.2,0.767200,0.888559,0.836495,0.624656,0.701138,Administrative Assistant,0.541531,0.720029
3,0.493143,9.59168,0.2,0.185888,0.732189,1.000000,0.959320,0.339626,Senior Manager,1.000000,0.757336
4,0.567230,0.00000,0.2,0.566706,0.817545,1.000000,0.677305,0.565582,Anonymous Employee,0.614825,0.726394


In [ ]:
# Drop the original categorical 'role' column
BO_data.drop(columns=['role'], inplace=True)

In [ ]:
# Question: How does job difficulty and Workeload combine to create mental strain?
# Pressure Index: High score indicates a high risk of exhaustion due to complex tasks plus heavy workload
BO_data['pressure_index'] = BO_data['role_complexity_score'] * BO_data['workload_score']

In [ ]:
# Question: What is the relationship between overtime hours and  satisfaction?
# Burnout Propensity: High score means the employee is working overtime but has low satisfaction
BO_data['burnout_propensity'] = BO_data['overtime_hours'] * (1 - BO_data['satisfaction_score'])

In [ ]:
# Question: Is the employee extra effort actually leading to results?
# Effort Efficiency: Low score indicates that overtime hours are NOT translating into goal achievement.
BO_data['effort_efficiency'] = BO_data['goal_achievement_rate'] / (BO_data['overtime_hours'] + 1)

In [ ]:
# High workload with low satisfaction leads to burnout
BO_data['burnout_pressure'] = BO_data['workload_score'] / (BO_data['satisfaction_score'] + 0.1)

In [ ]:
# Reordering columns to place the Target (burnout_score) at the very end.
cols = [c for c in BO_data.columns if c != 'burnout_score'] + ['burnout_score']
BO_data = BO_data[cols]

In [ ]:
BO_data.head()

,workload_score,overtime_hours,role_complexity_score,satisfaction_score,team_sentiment,career_progression_score,goal_achievement_rate,meeting_participation,role_encoded,pressure_index,burnout_propensity,effort_efficiency,burnout_pressure,burnout_score
0,0.758117,0.00000,0.2,0.623746,0.662335,1.000000,0.632482,0.492131,0.728387,0.151623,0.000000,0.632482,1.047490,0.866643
1,0.788416,0.00000,0.2,0.982556,0.934661,1.000000,0.538587,0.981394,0.756589,0.157683,0.000000,0.538587,0.728291,0.218996
2,0.697617,0.00000,0.2,0.767200,0.888559,0.836495,0.624656,0.701138,0.720029,0.139523,0.000000,0.624656,0.804447,0.541531
3,0.493143,9.59168,0.2,0.185888,0.732189,1.000000,0.959320,0.339626,0.757336,0.098629,7.808699,0.090573,1.724950,1.000000
4,0.567230,0.00000,0.2,0.566706,0.817545,1.000000,0.677305,0.565582,0.726394,0.113446,0.000000,0.677305,0.850794,0.614825


In [ ]:
# to Check Correlation matrix bettween the new features
BO_data[['pressure_index',
         'burnout_propensity',
         'effort_efficiency',
         'burnout_score']].corr()

,pressure_index,burnout_propensity,effort_efficiency,burnout_score
pressure_index,1.000000,-0.304819,0.430225,-0.245441
burnout_propensity,-0.304819,1.000000,-0.579646,0.302801
effort_efficiency,0.430225,-0.579646,1.000000,-0.330370
burnout_score,-0.245441,0.302801,-0.330370,1.000000


#Phase Four: Model training, Selection and Evaluation

## Overview:


## Key Objectives:


### Baseline model: Linear Regression


In [ ]:
# Abrar

### Model 1: Random Forest

In [ ]:
# Abrar

### Model 2: XGBoost

In [ ]:
# Abdulmajeed

### Model 3: LightGBM

In [ ]:
# Abdulmajeed